# Metrics, Feature and Score for Driving Problem

## Metrics
Each candidate driving policy is evaluated in SUMO and produces a dictionary of aggregated metrics per test case. 

In [ ]:
{
    'critical_ttc_count': 28,
    'collisions': 0,
    'emergencyStops': 0,
    'emergencyBraking': 4,
    'teleports': 0,
    'avg_fuel_consumption': 8.32,
    'avg_speed': 12.51,
    'speed_variance': 16.22
}

## Feature definitions for driving problem

Feature vectors are used for clustering solutions in the solution database (e.g., for MAP-Elites style diversity preservation). They define behavioral signatures rather than scalar quality alone. Multiple feature construction strategies are supported, allowing users to tailor diversity representation to specific problem requirements.

### Behavioral Signature (Recommended)

The behavioral signature maps aggregated metrics into interpretable categorical levels representing safety, efficiency, and smoothness. 

In [ ]:
def _get_feature_segmentation(test_metrics: MetricsPerTest) -> Signature:
    """
    Args:
        test_metrics (MetricsPerTest): A mapping of test metrics to a signature.
    
    Returns:
        Signature: (safety_level, speed_efficiency, fuel_efficiency, traffic_smoothness) on 0-3 scales.
        Level 0: terrible; 1: bad; 2: good; 3: perfect
    """
    # Aggregate metrics across all tests
    avg_test_metrics = average_test_metrics(test_metrics)
    
    # Safety Level: Higher = Safer
    if avg_test_metrics['collisions'] >= 1.0 or avg_test_metrics['teleports'] >= 1.0:
        safety_level = 0
    elif avg_test_metrics['emergencyStops'] >= 3.0:
        safety_level = 1
    elif avg_test_metrics['emergencyBraking'] >=  5.0 or avg_test_metrics['critical_ttc_count'] >= 50.0:
        safety_level = 2
    else:
        safety_level = 3
    
    # Speed efficiency (not too slow, not too fast is best)
    if 40 / 3.6 <= avg_test_metrics['avg_speed'] <= 50 / 3.6:
        speed_efficiency = 3
    elif 30 / 3.6 <= avg_test_metrics['avg_speed'] <= 40 / 3.6 or \
        50 / 3.6 <= avg_test_metrics['avg_speed'] <= 55 / 3.6:
        speed_efficiency = 2
    elif 15 / 3.6 <= avg_test_metrics['avg_speed'] <= 30 / 3.6:
        speed_efficiency = 1
    else:
        speed_efficiency = 0
    
    # Traffic smoothness (0-4): Higher = More smooth
    if avg_test_metrics['speed_variance'] <= 5:
        traffic_smoothness = 3
    elif avg_test_metrics['speed_variance'] <= 10:
        traffic_smoothness  = 2
    elif avg_test_metrics['speed_variance'] <= 20:
        traffic_smoothness  = 1
    else:
        traffic_smoothness  = 0
    
    return (safety_level, speed_efficiency, traffic_smoothness)

### Quantized signature
discretizes raw metrics into bins for higher-resolution clustering.

In [ ]:
def _get_feature_quantized(test_metrics: MetricsPerTest) -> Signature:
    """
    Quantized signature: Discretizes key metrics into bins.
    
    Returns tuple of quantized values for clustering.
    Each value represents a performance tier.
    """
    # Aggregate across tests
    total_collisions = sum(metrics.get('collisions', 0) for metrics in test_metrics.values())
    total_emergency_events = sum(
        metrics.get('emergencyStops', 0) + metrics.get('emergencyBraking', 0) 
        for metrics in test_metrics.values()
    )
    avg_fuel = sum(metrics.get('avg_fuel_consumption', 0) for metrics in test_metrics.values()) / len(test_metrics)
    avg_speed = sum(metrics.get('avg_speed', 0) for metrics in test_metrics.values()) / len(test_metrics)
    avg_speed_variance = sum(metrics.get('speed_variance', 0) for metrics in test_metrics.values()) / len(test_metrics)
    
    num_tests = len(test_metrics)
    
    # Quantize each metric (0-9 scale for good clustering resolution)
    collision_bin = min(9, total_collisions)
    emergency_bin = min(9, total_emergency_events // num_tests)
    fuel_bin = min(9, int(avg_fuel - 4))  # Assuming fuel range 4-13+
    speed_bin = min(9, int(avg_speed // 7))  # 0-7 km/h -> 0, 7-14 -> 1, etc.
    variance_bin = min(9, int(avg_speed_variance // 3))  # 0-3 -> 0, 3-6 -> 1, etc.
    
    return (collision_bin, emergency_bin, fuel_bin, speed_bin, variance_bin)

## Continuous signature
normalized real-valued features suitable for continuous clustering methods (e.g., k-means).

In [ ]:
def _get_feature_continuous(test_metrics: MetricsPerTest) -> Signature:
    """
    Continuous signature: Normalized continuous values.
    
    Good for algorithms that can handle continuous clustering (like k-means).
    Values are normalized to [0, 1] range.
    """
    # Aggregate metrics
    total_collisions = sum(metrics.get('collisions', 0) for metrics in test_metrics.values())
    total_emergency_stops = sum(metrics.get('emergencyStops', 0) for metrics in test_metrics.values())
    total_emergency_braking = sum(metrics.get('emergencyBraking', 0) for metrics in test_metrics.values())
    total_critical_ttc = sum(metrics.get('critical_ttc_count', 0) for metrics in test_metrics.values())
    
    avg_fuel = sum(metrics.get('avg_fuel_consumption', 0) for metrics in test_metrics.values()) / len(test_metrics)
    avg_speed = sum(metrics.get('avg_speed', 0) for metrics in test_metrics.values()) / len(test_metrics)
    avg_speed_variance = sum(metrics.get('speed_variance', 0) for metrics in test_metrics.values()) / len(test_metrics)
    
    num_tests = len(test_metrics)
    
    # Normalize metrics to [0, 1] range
    # Safety metrics (lower is better, so we invert)
    collision_norm = max(0, 1 - total_collisions / (num_tests * 2))  # Assume max 2 collisions/test
    emergency_norm = max(0, 1 - total_emergency_stops / (num_tests * 5))  # Assume max 5/test
    braking_norm = max(0, 1 - total_emergency_braking / (num_tests * 10))  # Assume max 10/test
    ttc_norm = max(0, 1 - total_critical_ttc / (num_tests * 50))  # Assume max 50/test
    
    # Performance metrics
    fuel_norm = max(0, 1 - (avg_fuel - 4) / 12)  # Assume range 4-16 L/100km
    speed_norm = min(1, avg_speed / 60)  # Normalize to max 60 km/h
    variance_norm = max(0, 1 - avg_speed_variance / 30)  # Assume max variance of 30
    
    return (collision_norm, emergency_norm, braking_norm, ttc_norm, 
            fuel_norm, speed_norm, variance_norm)

### Risk profile signature
emphasizes risk–performance trade-offs.

In [ ]:
def _get_feature_risk_profile(test_metrics: MetricsPerTest) -> Signature:
    """
    Risk Profile signature: Focuses on safety vs performance trade-offs.
    
    Returns (risk_level, performance_level) tuple.
    Useful for clustering algorithms by their risk-performance characteristics.
    """
    # Calculate risk indicators
    total_collisions = sum(metrics.get('collisions', 0) for metrics in test_metrics.values())
    total_emergency_events = sum(
        metrics.get('emergencyStops', 0) + 
        metrics.get('emergencyBraking', 0) + 
        metrics.get('teleports', 0)
        for metrics in test_metrics.values()
    )
    total_critical_ttc = sum(metrics.get('critical_ttc_count', 0) for metrics in test_metrics.values())
    
    # Calculate performance indicators
    avg_fuel = sum(metrics.get('avg_fuel_consumption', 0) for metrics in test_metrics.values()) / len(test_metrics)
    avg_speed = sum(metrics.get('avg_speed', 0) for metrics in test_metrics.values()) / len(test_metrics)
    avg_speed_variance = sum(metrics.get('speed_variance', 0) for metrics in test_metrics.values()) / len(test_metrics)
    
    num_tests = len(test_metrics)
    
    # Risk Level (0-10): Higher = More Risky
    risk_score = 0
    risk_score += total_collisions * 5  # Heavy weight for collisions
    risk_score += total_emergency_events * 0.5
    risk_score += total_critical_ttc * 0.1
    risk_level = min(10, int(risk_score / num_tests))
    
    # Performance Level (0-10): Higher = Better Performance
    # Combine speed efficiency and fuel efficiency
    speed_performance = min(10, int(avg_speed / 5))  # Up to 50 km/h
    fuel_performance = max(0, 10 - int((avg_fuel - 5) / 0.8))  # Better fuel = higher score
    smoothness_performance = max(0, 10 - int(avg_speed_variance / 2))
    
    performance_level = min(10, (speed_performance + fuel_performance + smoothness_performance) // 3)
    
    return (risk_level, performance_level)

## Score
While feature vectors maintain diversity, optimization is guided by a scalar fitness score. The scoring function prioritizes safety, followed by efficiency and smoothness.

In [ ]:
def test_metrics_to_scores(metrics_per_test: MetricsPerTest) -> ScoresPerTest:
    """
    Converts a mapping of test metrics to a mapping of test scores. 
    
    Higher scores are better. The fitness function prioritizes:
    1. Safety (no collisions, minimal emergency events)
    2. Efficiency (good speed)
    3. Smoothness (low speed variance)
    There aspects are modeled by safety_score, speed_score and smoothness_score respectively.
    safety_score/speed_score/smoothness_score are all in range [0, 100].
    The final score is weighted average of these three metrics.
    """
    scores = {}
    
    for test_id, metrics in metrics_per_test.items():
        # Extract metrics with defaults
        critical_ttc = metrics.get('critical_ttc_count', 0)
        collisions = metrics.get('collisions', 0)
        emergency_stops = metrics.get('emergencyStops', 0)
        emergency_braking = metrics.get('emergencyBraking', 0)
        teleports = metrics.get('teleports', 0)
        fuel_consumption = metrics.get('avg_fuel_consumption', 0)
        avg_speed = metrics.get('avg_speed', 0)
        speed_variance = metrics.get('speed_variance', 0)
        
        # Safety Score - Most Important
        # Severe penalties for critical safety events
        safety_score = 100.0
        safety_score -= collisions * 50          # -50 per collision
        safety_score -= emergency_stops * 5     # -5 per emergency stop
        safety_score -= teleports * 30          # -30 per teleport (simulation failure)
        safety_score -= emergency_braking * 2   # -2 per emergency brake
        safety_score -= critical_ttc * 0.5      # -0.5 per critical time-to-collision
        #safety_score = max(0, safety_score)     # Ensure non-negative
        
        # Speed Efficiency Score (0-100)
        # Optimal speed range: 30-50 km/h for urban driving
        # You may need to adjust these ranges based on your simulation
        target_speed = 13.89  # m/s
        speed_deviation = abs(avg_speed - target_speed)
        if speed_deviation <= 1.5:  # close to target speed
            speed_score = 100
        elif speed_deviation <= 3.5:  # still feel good
            speed_score = 100 - (speed_deviation - 1.5) * 5
        else:
            speed_score = max(0, 90 - (speed_deviation - 3.5) * 8.662)  # reduced to zero at zero speed
        
        # Smoothness Score
        # Lower speed variance indicates smoother driving
        if speed_variance <= 5.0:
            smoothness_score = 100
        elif speed_variance <= 10.0:
            smoothness_score = 100 - (speed_variance - 5.0) * 4
        else:
            smoothness_score = 80 - (speed_variance - 10.0) * 8  # reduced to zero at 20.0
            smoothness_score = max(0, 80 - (speed_variance - 10.0) * 8)  # reduced to zero at 20.0
        
        # Weighted Combined Score
        # Safety is most important, followed by efficiency and smoothness
        weights = {
            'safety': 0.5,      # Most critical
            'speed': 0.3,       # Speed efficiency
            'smoothness': 0.2  # Driving smoothness
        }
        
        combined_score = (
            weights['safety'] * safety_score + 
            weights['speed'] * speed_score + 
            weights['smoothness'] * smoothness_score
        )
        
        scores[test_id] = combined_score
    
    return scores